## Initialization


In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

## Read Bronze tables

In [0]:
df = spark.table("workspace.bronze.crm_cust_info")

## Silver Transformation

### Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

### Normalization

In [0]:
df = (
    df
    .withColumn(
        "cst_marital_status", 
        F.when(F.upper(col("cst_marital_status")) == "S", "Single")
         .when(F.upper(col("cst_marital_status")) ==  "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(col("cst_gndr")) == "M", "Male")
         .when(F.upper(col("cst_gndr")) == "F", "Female")
         .otherwise("n/a")
    )
)

### Remove records with missing customer ID

In [0]:
df = df.filter(col("cst_id").isNotNull())

### Renaming columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "firstname",
    "cst_lastname": "lastname",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}

for old, new in RENAME_MAP.items():
    df = df.withColumnRenamed(old, new)

### Sanity checks of dataframe

In [0]:
df.limit(10).display()

customer_id,customer_key,firstname,lastname,marital_status,gender,created_date
11000,AW00011000,Jon,Yang,Married,Male,2025-10-06
11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06
11002,AW00011002,Ruben,Torres,Married,Male,2025-10-06
11003,AW00011003,Christy,Zhu,Single,Female,2025-10-06
11004,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06
11005,AW00011005,Julio,Ruiz,Single,Male,2025-10-06
11006,AW00011006,Janet,Alvarez,Single,Female,2025-10-06
11007,AW00011007,Marco,Mehta,Married,Male,2025-10-06
11008,AW00011008,Rob,Verhoff,Single,Female,2025-10-06
11009,AW00011009,Shannon,Carlson,Single,Male,2025-10-06


### Writing silver tables

In [0]:
df = df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")

### Sanity checks of silver tables

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10

customer_id,customer_key,firstname,lastname,marital_status,gender,created_date
11000,AW00011000,Jon,Yang,Married,Male,2025-10-06
11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06
11002,AW00011002,Ruben,Torres,Married,Male,2025-10-06
11003,AW00011003,Christy,Zhu,Single,Female,2025-10-06
11004,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06
11005,AW00011005,Julio,Ruiz,Single,Male,2025-10-06
11006,AW00011006,Janet,Alvarez,Single,Female,2025-10-06
11007,AW00011007,Marco,Mehta,Married,Male,2025-10-06
11008,AW00011008,Rob,Verhoff,Single,Female,2025-10-06
11009,AW00011009,Shannon,Carlson,Single,Male,2025-10-06
